## ExPECA HI GPU Server + Raspberry Pi Edge Device Setup

This notebook creates the public-IP ExPECA setup for the GPU benchmark:

- **Edge server**: `worker-05`, GPU node, CUDA container, public IP, port `8001`.
- **Edge device**: `worker-21`, Raspberry Pi / ARM64 container, public IP, port `8000`.
- **Controller**: your laptop, via `src/run_thesis_reproduction.py`.

Use this notebook after building and pushing:

- GPU edge-server image, e.g. `<namespace>/hi-framework-edge-server:gpu-amd64-001`
- ARM64 edge-device image, e.g. `<namespace>/hi-framework-edge-device:cpu-arm64-001`

At the end, copy the printed IPs into `config/experiment.env` and run the GPU benchmark.


### 1. Authentication

Download your OpenRC file from the ExPECA/Chameleon API Access page and keep it in the repository root or next to this notebook.

The next cell tries to find `*-openrc.sh` automatically. If needed, set `OPENRC_PATH` manually.


In [1]:
import os, re
from pathlib import Path
from getpass import getpass

OPENRC_PATH = None  # Example: "justin-project-openrc.sh"

candidate_paths = []
if OPENRC_PATH:
    candidate_paths.append(Path(OPENRC_PATH))
candidate_paths.extend(Path.cwd().glob("*-openrc.sh"))
candidate_paths.extend(Path.cwd().parent.glob("*-openrc.sh"))

openrc_path = next((path for path in candidate_paths if path.exists()), None)
if openrc_path is None:
    raise FileNotFoundError(
        "Could not find an OpenRC file. Set OPENRC_PATH to your *-openrc.sh file."
    )

script_content = openrc_path.read_text()
pattern = r'export\s+(\w+)\s*=\s*("[^"]+"|[^"\n]+)'
for name, value in re.findall(pattern, script_content):
    os.environ[name] = value.strip('"')

os.environ["OS_PASSWORD"] = getpass("enter your ExPECA password: ")
print(f"Loaded OpenRC environment from: {openrc_path}")


Loaded OpenRC environment from: /home/justin/KTH/Sacnia_Internship/Batch-Hierarchical-Inference-GPU/notebooks/iustin-project-openrc.sh


### 2. Imports, Repo Config, And Helpers

Install notebook dependencies from the project terminal before running this cell:

```bash
scripts/setup_expeca_notebook_env.sh
```

Then restart the notebook kernel and continue here.


In [2]:
import json
import os
import time
from pathlib import Path

import requests
from loguru import logger
import chi.network
import chi.container
from chi.expeca import (
    reserve,
    list_reservations,
    get_available_publicips,
    get_worker_interfaces,
)


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / "config" / "experiment.env").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current notebook path.")


REPO_ROOT = find_repo_root()
print(f"Repository root: {REPO_ROOT}")


def load_env_file(path: Path) -> dict[str, str]:
    values = {}
    if not path.exists():
        return values
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


config = load_env_file(REPO_ROOT / "config" / "defaults.env")
config.update(load_env_file(REPO_ROOT / "config" / "experiment.env"))

EXPECA_IMAGE_NAMESPACE = config.get("EXPECA_IMAGE_NAMESPACE", "YOUR_DOCKERHUB_NAMESPACE")
GPU_EDGE_SERVER_IMAGE_TAG = config.get("EXPECA_GPU_EDGE_SERVER_IMAGE_TAG", "gpu-amd64-001")
ARM64_EDGE_DEVICE_IMAGE_TAG = config.get("EXPECA_EDGE_DEVICE_ARM64_IMAGE_TAG", "cpu-arm64-001")

EDGE_SERVER_IMAGE = config.get(
    "EXPECA_GPU_EDGE_SERVER_IMAGE",
    f"{EXPECA_IMAGE_NAMESPACE}/hi-framework-edge-server:{GPU_EDGE_SERVER_IMAGE_TAG}",
)
EDGE_DEVICE_ARM64_IMAGE = config.get(
    "EXPECA_EDGE_DEVICE_ARM64_IMAGE",
    f"{EXPECA_IMAGE_NAMESPACE}/hi-framework-edge-device:{ARM64_EDGE_DEVICE_IMAGE_TAG}",
)

EDGE_SERVER_CONTAINER_NAME = "hi-gpu-edge-server"
EDGE_DEVICE_CONTAINER_NAME = "hi-gpu-edge-device"
GPU_WORKER_NAME = "worker-05"
RASPBERRY_PI_WORKER_NAME = "worker-21"
SERVER_PUBLIC_NETWORK = "serverpublic"
PUBLIC_GATEWAY_IP = "130.237.11.97"
DNS_IP = "8.8.8.8"
ROOT_PASSWORD = "expeca"

print("GPU edge-server image:", EDGE_SERVER_IMAGE)
print("ARM64 edge-device image:", EDGE_DEVICE_ARM64_IMAGE)
print("GPU worker:", GPU_WORKER_NAME)
print("Raspberry Pi worker:", RASPBERRY_PI_WORKER_NAME)


Repository root: /home/justin/KTH/Sacnia_Internship/Batch-Hierarchical-Inference-GPU
GPU edge-server image: justin157/hi-framework-edge-server:gpu-amd64-001
ARM64 edge-device image: justin157/hi-framework-edge-device:cpu-arm64-001
GPU worker: worker-05
Raspberry Pi worker: worker-21


In [3]:
INACTIVE_CONTAINER_STATUSES = {"Deleted", "Error"}


def containers_named(container_name, include_inactive=True):
    matches = [container for container in chi.container.list_containers() if container.name == container_name]
    if include_inactive:
        return matches
    return [container for container in matches if container.status not in INACTIVE_CONTAINER_STATUSES]


def print_container_summary(container):
    print(
        "name=", container.name,
        "uuid=", container.uuid,
        "status=", container.status,
        "reason=", getattr(container, "status_reason", None),
        "image=", getattr(container, "image", None),
        "addresses=", getattr(container, "addresses", None),
    )


def print_named_containers(container_name):
    matches = containers_named(container_name)
    if not matches:
        print(f"No containers named {container_name}.")
        return matches
    print(f"Containers named {container_name}:")
    for container in matches:
        print_container_summary(container)
    return matches


def wait_for_containers_removed(container_uuids, poll_interval=5, max_wait_seconds=180):
    deadline = time.time() + max_wait_seconds
    remaining = set(container_uuids)
    while remaining:
        existing = {
            container.uuid
            for container in chi.container.list_containers()
            if container.status not in INACTIVE_CONTAINER_STATUSES
        }
        remaining = remaining & existing
        if not remaining:
            print("Requested containers are removed or inactive.")
            return
        if time.time() >= deadline:
            raise TimeoutError(f"Timed out waiting for removal: {sorted(remaining)}")
        print("Waiting for removal:", sorted(remaining))
        time.sleep(poll_interval)


def destroy_named_containers(container_name, statuses=None, wait=True):
    matches = containers_named(container_name)
    if statuses is not None:
        matches = [container for container in matches if container.status in statuses]
    if not matches:
        print(f"No matching containers to destroy for {container_name}.")
        return []
    destroyed = []
    for container in matches:
        print("Destroying", container.name, container.uuid, container.status)
        chi.container.destroy_container(container.uuid)
        destroyed.append(container.uuid)
    if wait:
        wait_for_containers_removed(destroyed)
    return destroyed


def inspect_named_container(container_name, include_logs=True):
    matches = print_named_containers(container_name)
    active = [container for container in matches if container.status not in INACTIVE_CONTAINER_STATUSES]
    if len(active) != 1:
        if len(active) > 1:
            print("Multiple active matches exist. Clean up duplicates before continuing.")
        return matches
    container = active[0]
    if include_logs:
        print("logs:")
        try:
            print(chi.container.get_logs(container.uuid))
        except Exception as exc:
            print("Could not fetch logs yet:", repr(exc))
    return matches


def reserve_active_worker(worker_name, days=7, hours=0):
    leaseslist = list_reservations(brief=True)
    for lease_dict in leaseslist:
        if lease_dict["name"] == worker_name + "-lease" and lease_dict.get("status") == "ACTIVE":
            print(f"Using active lease for {worker_name}: {lease_dict['reservation_id']}")
            print(json.dumps(leaseslist, indent=4))
            return lease_dict["reservation_id"]

    print(f"Creating new lease for {worker_name}.")
    worker_lease = reserve(
        {"type": "device", "name": worker_name, "duration": {"days": days, "hours": hours}}
    )
    reservation_id = worker_lease["reservations"][0]["id"]
    print(f"Created lease for {worker_name}: {reservation_id}")
    print(json.dumps(list_reservations(brief=True), indent=4))
    return reservation_id


def available_worker_interface(worker_name):
    interfaces = list(get_worker_interfaces(worker_name).values())[0]
    available_ifs = [
        interface
        for interface, data in interfaces.items()
        if len(data.get("connections", [])) == 0
    ]
    logger.info(f"Available interfaces on {worker_name}: {available_ifs}")
    if not available_ifs:
        print(json.dumps(interfaces, indent=4))
        raise RuntimeError(f"No available interfaces on {worker_name}.")
    return available_ifs[0]


def choose_public_ip(index=0):
    available_pub_ips = get_available_publicips()
    if index >= len(available_pub_ips):
        raise IndexError(f"Requested public IP index {index}, but only got {available_pub_ips}")
    pub_ip = available_pub_ips[index]
    logger.info(f"Available public IPs: {available_pub_ips}")
    logger.info(f"Selected public IP: {pub_ip}")
    return pub_ip


def create_container_with_progress(container_name, create_kwargs, poll_interval=15, max_wait_seconds=1200):
    active_existing = containers_named(container_name, include_inactive=False)
    if active_existing:
        print_named_containers(container_name)
        raise RuntimeError(
            f"Active container named {container_name} already exists. "
            "Skip creation or destroy it intentionally first."
        )

    try:
        container = chi.container.create_container(
            name=container_name,
            start_timeout=60,
            **create_kwargs,
        )
        container_ref = container.uuid
    except TimeoutError:
        logger.warning(f"{container_name} is still creating after 60s; polling status now.")
        active = containers_named(container_name, include_inactive=False)
        if len(active) != 1:
            print_named_containers(container_name)
            raise RuntimeError(f"Could not find exactly one active {container_name} after timeout.")
        container_ref = active[0].uuid
    except Exception as exc:
        active = containers_named(container_name, include_inactive=False)
        if len(active) == 1:
            logger.warning(f"create_container returned {type(exc).__name__}: {exc}. Polling active container.")
            container_ref = active[0].uuid
        else:
            print_named_containers(container_name)
            raise

    deadline = time.time() + max_wait_seconds
    while True:
        container = chi.container.get_container(container_ref)
        reason = getattr(container, "status_reason", None)
        addresses = getattr(container, "addresses", None)
        print(time.strftime("%H:%M:%S"), "status:", container.status, "reason:", reason, "addresses:", addresses)
        if container.status == "Running":
            logger.success(f"{container_name} is running.")
            return container
        if container.status in INACTIVE_CONTAINER_STATUSES:
            raise RuntimeError(f"{container_name} failed with status {container.status}: {reason}")
        if time.time() >= deadline:
            raise TimeoutError(f"Timed out waiting for {container_name}; last status was {container.status}")
        time.sleep(poll_interval)


def check_http_endpoint(url):
    response = requests.get(url, timeout=15)
    print(url, "->", response.status_code)
    print(response.text[-1000:])
    response.raise_for_status()


### 3. Cleanup / Inspect Existing Containers

Run this before creating containers. Uncomment destroy calls only when you intentionally want to remove existing containers.


In [4]:
print_named_containers(EDGE_SERVER_CONTAINER_NAME)
print_named_containers(EDGE_DEVICE_CONTAINER_NAME)

# Common recovery after interrupted runs: remove stale failed records.
# destroy_named_containers(EDGE_SERVER_CONTAINER_NAME, statuses={"Error", "Creating"})
# destroy_named_containers(EDGE_DEVICE_CONTAINER_NAME, statuses={"Error", "Creating"})

# Full reset: use only when you intentionally want to recreate everything.
# destroy_named_containers(EDGE_DEVICE_CONTAINER_NAME)
# destroy_named_containers(EDGE_SERVER_CONTAINER_NAME)


No containers named hi-gpu-edge-server.
No containers named hi-gpu-edge-device.


[]

### 4. Reserve GPU Edge Server Worker (`worker-05`)


In [5]:
gpu_worker_reservation_id = reserve_active_worker(GPU_WORKER_NAME)


2026-07-16 00:08:33.326 | INFO     | chi.expeca:reserve:243 - reserving worker-05


Creating new lease for worker-05.


2026-07-16 00:08:35.636 | INFO     | chi.expeca:wait_until_lease_status:138 - waiting 120 seconds for worker-05-lease with id a5d0344c-3a7a-4722-b0ff-44d9b0cc9c29 to become "ACTIVE"
2026-07-16 00:08:40.709 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-05-lease with id a5d0344c-3a7a-4722-b0ff-44d9b0cc9c29 is PENDING.
2026-07-16 00:08:45.759 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-05-lease with id a5d0344c-3a7a-4722-b0ff-44d9b0cc9c29 is PENDING.
2026-07-16 00:08:50.802 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-05-lease with id a5d0344c-3a7a-4722-b0ff-44d9b0cc9c29 is PENDING.
2026-07-16 00:08:55.884 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-05-lease with id a5d0344c-3a7a-4722-b0ff-44d9b0cc9c29 is PENDING.
2026-07-16 00:09:00.942 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-05-lease with id a5d0344c-3a7a-4722-b0ff-44d9b0cc9c29 is PENDING.
2026-07-16 00:09:06.022 | INFO   

Created lease for worker-05: 667676f0-a112-4742-afe4-110d14ffb3dd
[
    {
        "name": "worker-08-lease",
        "id": "3552b00d-7a32-4ed7-8de7-33708a5e02e3",
        "reservation_id": "5de3e32e-be11-4cd0-8506-3062eabe2b6b",
        "status": "ACTIVE",
        "end_date": "2026-07-16T07:19:00.000000"
    },
    {
        "name": "worker-09-lease",
        "id": "828a2064-9aad-444d-bf44-a469b545881e",
        "reservation_id": "863b5f4a-0bf2-4cd6-8772-4e6979c77481",
        "status": "ACTIVE",
        "end_date": "2026-07-20T15:45:00.000000"
    },
    {
        "name": "worker-09-lease",
        "id": "9fdcf126-cd12-4d1b-9a76-871fd77b2ecf",
        "reservation_id": "80887ec6-2bff-425e-9e5f-29c6c65295e8",
        "status": "TERMINATED",
        "end_date": "2026-07-13T14:41:00.000000"
    },
    {
        "name": "worker-05-lease",
        "id": "a5d0344c-3a7a-4722-b0ff-44d9b0cc9c29",
        "reservation_id": "667676f0-a112-4742-afe4-110d14ffb3dd",
        "status": "ACTIVE",
    

### 5. Create GPU Edge Server Container

This follows the official ExPECA GPU public-container example:

- `runtime="nvidia"`
- `resources.limits.nvidia_com_gpu="1"`
- public network `serverpublic`

If the first public IP fails, change `SERVER_PUBLIC_IP_INDEX` to another index and rerun after cleanup.


In [6]:
SERVER_PUBLIC_IP_INDEX = 0

server_pub_ip = choose_public_ip(SERVER_PUBLIC_IP_INDEX)
server_interface = available_worker_interface(GPU_WORKER_NAME)
publicnet = chi.network.get_network(SERVER_PUBLIC_NETWORK)

server_container = create_container_with_progress(
    EDGE_SERVER_CONTAINER_NAME,
    {
        "image": EDGE_SERVER_IMAGE,
        "reservation_id": gpu_worker_reservation_id,
        "runtime": "nvidia",
        "environment": {
            "DNS_IP": DNS_IP,
            "GATEWAY_IP": PUBLIC_GATEWAY_IP,
            "PASS": ROOT_PASSWORD,
            "DEVICE": "cuda",
        },
        "mounts": [],
        "nets": [{"network": publicnet["id"]}],
        "labels": {
            "networks.1.interface": server_interface,
            "networks.1.ip": server_pub_ip + "/27",
            "networks.1.gateway": PUBLIC_GATEWAY_IP,
            "capabilities.privileged": "true",
            "resources.limits.nvidia_com_gpu": "1",
        },
    },
)

edge_server_pub_ip = server_pub_ip
logger.success(f"GPU Edge Server public IP: {edge_server_pub_ip}")


2026-07-16 00:10:24.533 | INFO     | __main__:choose_public_ip:123 - Available public IPs: ['130.237.11.114', '130.237.11.123', '130.237.11.125', '130.237.11.126']
2026-07-16 00:10:24.534 | INFO     | __main__:choose_public_ip:124 - Selected public IP: 130.237.11.114
2026-07-16 00:10:34.939 | INFO     | __main__:available_worker_interface:111 - Available interfaces on worker-05: ['eno3np0', 'eno4np1', 'ens1f1']
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
2026-07-16 00:11:39.354 | WARNING  | __main__:create_container_with_progress:145 - hi-gpu-edge-server is still creating after 60s; polling status now.


00:11:39 status: Creating reason: None addresses: {}
00:11:54 status: Creating reason: None addresses: {}
00:12:10 status: Creating reason: None addresses: {}
00:12:25 status: Creating reason: None addresses: {}
00:12:40 status: Creating reason: None addresses: {}
00:12:55 status: Creating reason: None addresses: {}
00:13:10 status: Creating reason: None addresses: {}
00:13:25 status: Creating reason: None addresses: {}


2026-07-16 00:13:41.268 | SUCCESS  | __main__:create_container_with_progress:167 - hi-gpu-edge-server is running.
2026-07-16 00:13:41.269 | SUCCESS  | __main__:<module>:32 - GPU Edge Server public IP: 130.237.11.114


00:13:41 status: Running reason: None addresses: {'e9666f9c-1ef6-4be7-81cd-32373cfacc28': [{'addr': '192.168.123.26', 'version': 4, 'port': '9c0e8fbe-1019-4201-ba27-021da4c65b56'}]}


In [7]:
inspect_named_container(EDGE_SERVER_CONTAINER_NAME)
check_http_endpoint(f"http://{edge_server_pub_ip}:8001/logs")


Containers named hi-gpu-edge-server:
name= hi-gpu-edge-server uuid= 75f9b7b0-458d-47e3-9aac-ddbb7c2065b8 status= Running reason= None image= justin157/hi-framework-edge-server:gpu-amd64-001 addresses= {'e9666f9c-1ef6-4be7-81cd-32373cfacc28': [{'addr': '192.168.123.26', 'version': 4, 'port': '9c0e8fbe-1019-4201-ba27-021da4c65b56'}]}
logs:
[2026-07-15 22:13:24] Appending custom DNS: 8.8.8.8
[2026-07-15 22:13:24] Setting default gateway to 130.237.11.97
[2026-07-15 22:13:24] Configuring SSH access
Starting OpenBSD Secure Shell server: sshd.
[2026-07-15 22:13:24] Starting edge_server.py
2026-07-15 22:13:27,306 - INFO - Deleted file: results/EdgeServer.log
2026-07-15 22:13:27,306 - INFO - Edge Server: Starting... (Device: cuda, Port: 8001)
2026-07-15 22:13:27,323 - INFO - Edge Server: Runtime diagnostics: {"torch_version": "2.8.0+cu128", "cuda_available": true, "cuda_version": "12.8", "selected_device": "cuda", "cuda_device_index": 0, "cuda_device_name": "NVIDIA L4", "cuda_device_capability

### 6. Optional GPU Sanity Check

The edge-server logs include runtime diagnostics from PyTorch. This is more reliable than shelling out through ExPECA command execution.


In [ ]:
logs = requests.get(f"http://{edge_server_pub_ip}:8001/logs", timeout=20).text
diagnostic_lines = [line for line in logs.splitlines() if "Runtime diagnostics" in line]
if not diagnostic_lines:
    raise RuntimeError("No runtime diagnostics found in edge-server logs yet.")

print(diagnostic_lines[-1])
if '"cuda_available": true' not in diagnostic_lines[-1]:
    raise RuntimeError("CUDA is not reported as available by the edge-server container.")


### 7. Reserve Raspberry Pi Edge Device Worker (`worker-21`)


In [9]:
pi_worker_reservation_id = reserve_active_worker(RASPBERRY_PI_WORKER_NAME)


2026-07-16 00:14:19.148 | INFO     | chi.expeca:reserve:243 - reserving worker-21


Creating new lease for worker-21.


2026-07-16 00:14:21.856 | INFO     | chi.expeca:wait_until_lease_status:138 - waiting 120 seconds for worker-21-lease with id c5afb3ad-911b-4803-9419-9d54c63c2a75 to become "ACTIVE"
2026-07-16 00:14:26.925 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-21-lease with id c5afb3ad-911b-4803-9419-9d54c63c2a75 is PENDING.
2026-07-16 00:14:32.002 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-21-lease with id c5afb3ad-911b-4803-9419-9d54c63c2a75 is PENDING.
2026-07-16 00:14:37.052 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-21-lease with id c5afb3ad-911b-4803-9419-9d54c63c2a75 is PENDING.
2026-07-16 00:14:42.114 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-21-lease with id c5afb3ad-911b-4803-9419-9d54c63c2a75 is PENDING.
2026-07-16 00:14:47.173 | INFO     | chi.expeca:wait_until_lease_status:145 - lease worker-21-lease with id c5afb3ad-911b-4803-9419-9d54c63c2a75 is PENDING.
2026-07-16 00:14:52.219 | INFO   

Created lease for worker-21: 6a39b7f7-b745-492d-bd0c-64fe6614f119
[
    {
        "name": "worker-08-lease",
        "id": "3552b00d-7a32-4ed7-8de7-33708a5e02e3",
        "reservation_id": "5de3e32e-be11-4cd0-8506-3062eabe2b6b",
        "status": "ACTIVE",
        "end_date": "2026-07-16T07:19:00.000000"
    },
    {
        "name": "worker-09-lease",
        "id": "828a2064-9aad-444d-bf44-a469b545881e",
        "reservation_id": "863b5f4a-0bf2-4cd6-8772-4e6979c77481",
        "status": "ACTIVE",
        "end_date": "2026-07-20T15:45:00.000000"
    },
    {
        "name": "worker-09-lease",
        "id": "9fdcf126-cd12-4d1b-9a76-871fd77b2ecf",
        "reservation_id": "80887ec6-2bff-425e-9e5f-29c6c65295e8",
        "status": "TERMINATED",
        "end_date": "2026-07-13T14:41:00.000000"
    },
    {
        "name": "worker-05-lease",
        "id": "a5d0344c-3a7a-4722-b0ff-44d9b0cc9c29",
        "reservation_id": "667676f0-a112-4742-afe4-110d14ffb3dd",
        "status": "ACTIVE",
    

### 8. Create Raspberry Pi Edge Device Container

This uses the ARM64 edge-device image and points it at the GPU server public IP.

If the first public IP fails, change `DEVICE_PUBLIC_IP_INDEX` to another index and rerun after cleanup.


In [10]:
DEVICE_PUBLIC_IP_INDEX = 0

if "YOUR_DOCKERHUB_NAMESPACE" in EDGE_DEVICE_ARM64_IMAGE:
    raise ValueError(
        "EDGE_DEVICE_ARM64_IMAGE is not configured. Build/push an ARM64 edge-device image "
        "and set EXPECA_EDGE_DEVICE_ARM64_IMAGE or EXPECA_EDGE_DEVICE_ARM64_IMAGE_TAG."
    )

device_pub_ip = choose_public_ip(DEVICE_PUBLIC_IP_INDEX)
device_interface = available_worker_interface(RASPBERRY_PI_WORKER_NAME)
publicnet = chi.network.get_network(SERVER_PUBLIC_NETWORK)

device_container = create_container_with_progress(
    EDGE_DEVICE_CONTAINER_NAME,
    {
        "image": EDGE_DEVICE_ARM64_IMAGE,
        "reservation_id": pi_worker_reservation_id,
        "environment": {
            "DNS_IP": DNS_IP,
            "GATEWAY_IP": PUBLIC_GATEWAY_IP,
            "PASS": ROOT_PASSWORD,
            "DEVICE": "cpu",
            "EDGE_SERVER_IP": edge_server_pub_ip,
        },
        "mounts": [],
        "nets": [{"network": publicnet["id"]}],
        "labels": {
            "networks.1.interface": device_interface,
            "networks.1.ip": device_pub_ip + "/27",
            "networks.1.gateway": PUBLIC_GATEWAY_IP,
            "capabilities.privileged": "true",
        },
    },
)

edge_device_pub_ip = device_pub_ip
logger.success(f"Raspberry Pi Edge Device public IP: {edge_device_pub_ip}")
logger.success(f"GPU Edge Server public IP: {edge_server_pub_ip}")


2026-07-16 00:16:11.860 | INFO     | __main__:choose_public_ip:123 - Available public IPs: ['130.237.11.123', '130.237.11.125', '130.237.11.126']
2026-07-16 00:16:11.861 | INFO     | __main__:choose_public_ip:124 - Selected public IP: 130.237.11.123
2026-07-16 00:16:25.247 | INFO     | __main__:available_worker_interface:111 - Available interfaces on worker-21: ['enx000acd47c9fc', 'enx000acd47cdaf', 'enx000acd47cdb0', 'eth0']
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
2026-07-16 00:17:30.182 | WARNING  | __main__:create_container_with_progress:145 - hi-gpu-edge-device is still creating after 60s; polling status now.


00:17:30 status: Creating reason: None addresses: {}
00:17:45 status: Creating reason: None addresses: {}
00:18:00 status: Creating reason: None addresses: {}
00:18:16 status: Creating reason: None addresses: {}
00:18:31 status: Creating reason: None addresses: {}
00:18:46 status: Creating reason: None addresses: {}
00:19:01 status: Creating reason: None addresses: {}
00:19:17 status: Creating reason: None addresses: {}
00:19:32 status: Creating reason: None addresses: {}
00:19:47 status: Creating reason: None addresses: {}
00:20:02 status: Creating reason: None addresses: {}
00:20:17 status: Creating reason: None addresses: {}
00:20:33 status: Creating reason: None addresses: {}
00:20:48 status: Creating reason: None addresses: {}
00:21:03 status: Creating reason: None addresses: {}
00:21:18 status: Creating reason: None addresses: {}
00:21:34 status: Creating reason: None addresses: {}
00:21:49 status: Creating reason: None addresses: {}
00:22:04 status: Creating reason: None address

2026-07-16 00:22:19.672 | SUCCESS  | __main__:create_container_with_progress:167 - hi-gpu-edge-device is running.
2026-07-16 00:22:19.672 | SUCCESS  | __main__:<module>:37 - Raspberry Pi Edge Device public IP: 130.237.11.123
2026-07-16 00:22:19.673 | SUCCESS  | __main__:<module>:38 - GPU Edge Server public IP: 130.237.11.114


00:22:19 status: Running reason: None addresses: {'e9666f9c-1ef6-4be7-81cd-32373cfacc28': [{'addr': '192.168.105.230', 'version': 4, 'port': '368e49f4-1de9-4035-ae6c-06cd3ad67b2b'}]}


In [11]:
inspect_named_container(EDGE_DEVICE_CONTAINER_NAME)
check_http_endpoint(f"http://{edge_device_pub_ip}:8000/logs")


Containers named hi-gpu-edge-device:
name= hi-gpu-edge-device uuid= 349b97e4-e589-4a66-bae3-a295201ce29d status= Running reason= None image= justin157/hi-framework-edge-device:cpu-arm64-001 addresses= {'e9666f9c-1ef6-4be7-81cd-32373cfacc28': [{'addr': '192.168.105.230', 'version': 4, 'port': '368e49f4-1de9-4035-ae6c-06cd3ad67b2b'}]}
logs:
[2026-07-15 22:22:15] Appending custom DNS: 8.8.8.8
[2026-07-15 22:22:15] Setting default gateway to 130.237.11.97
[2026-07-15 22:22:15] Configuring SSH access
Starting OpenBSD Secure Shell server: sshd.
[2026-07-15 22:22:15] Starting edge_device.py
2026-07-15 22:22:32,482 - INFO - Deleted file: results/EdgeDevice.log
2026-07-15 22:22:32,485 - INFO - Edge Device: Starting... (Device: cpu, Port: 8000)
INFO:     Started server process [1]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)

http://130.237.11.123:8000/logs -> 200
2026-07-15 22:22:32,485

### 9. Copy These Values To `config/experiment.env`

After both endpoints return `200 OK`, put these values in `config/experiment.env`.


In [12]:
print("Update config/experiment.env with:")
print("DEVICE=cuda")
print("EXPECA_EDGE_SERVER_DEVICE=cuda")
print(f"EDGE_SERVER_IP={edge_server_pub_ip}")
print(f"EDGE_DEVICE_IP={edge_device_pub_ip}")
print("THESIS_CONFIGS_TO_RUN=002,003,004,005,006,007")
print("LML_BATCHING_MODE=auto")
print("LML_INITIAL_BATCH_SIZE=16")
print("LML_MIN_BATCH_SIZE=1")
print("LML_MAX_BATCH_SIZE=256")
print("LML_GPU_MEMORY_FRACTION=0.9")
print("LML_OOM_RETRY=true")


Update config/experiment.env with:
DEVICE=cuda
EXPECA_EDGE_SERVER_DEVICE=cuda
EDGE_SERVER_IP=130.237.11.114
EDGE_DEVICE_IP=130.237.11.123
THESIS_CONFIGS_TO_RUN=002,003,004,005,006,007
LML_BATCHING_MODE=auto
LML_INITIAL_BATCH_SIZE=16
LML_MIN_BATCH_SIZE=1
LML_MAX_BATCH_SIZE=256
LML_GPU_MEMORY_FRACTION=0.9
LML_OOM_RETRY=true


### 10. Run The GPU Benchmark From The Terminal

From the repository root:

```bash
.venv/bin/python src/run_thesis_reproduction.py --dry-run
.venv/bin/python src/run_thesis_reproduction.py
```

Expected output folder:

```text
results/thesis_reproduction_gpu/
```
